In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Create and move into a specific project folder
import os
project_folder = '/content/drive/MyDrive/Colab_Projects/RunningLMM'

if not os.path.exists(project_folder):
    os.makedirs(project_folder)
    print(f"Created folder: {project_folder}")

# 3. Change the working directory to this folder
os.chdir(project_folder)
print(f"Current Directory: {os.getcwd()}")

ValueError: mount failed

In [ ]:
# Run this in a cell
!nvidia-smi

Fri Jan 16 15:47:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - && \
sudo apt-get install -y nodejs && \
sudo npm install -g @anthropic-ai/claude-code && \
export PATH=/usr/bin:$PATH

2026-01-16 15:47:13 - Installing pre-requisites
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,297 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,869 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,615 kB]
Get:13 

In [ ]:
!claude

Welcome to Claude Code v2.1.7 
…………………………………………………………………………………………………………………………………………………………

     *                                       █████▓▓░
                                 *         ███▓░     ░░
            ░░░░░░                        ███▓░
    ░░░   ░░░░░░░░░░                      ███▓░
   ░░░░░░░░░░░░░░░░░░░    *                ██▓░░      ▓
                                             ░▓▓███▓▓░
 *                                 ░░░░                   
                                 ░░░░░░░░                 
                               ░░░░░░░░░░░░░░░░           
       █████████                                        *
      ██▄█████▄██                        *
       █████████      *
…………………█ █   █ █………………………………………………………………………………………………………………


 Let's get started.

 Choose the text style that looks best with your terminal
 To change this later, run /theme

 ❯ 1. Dark mode ✔
   2. Light mode
   3. Dark mode (colorblind-friendly)
   4. Light mode (colorblind-friendly)
   

In [ ]:
# Install required packages
!pip install -q transformers torch datasets accelerate tqdm huggingface_hub bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.9 MB/s eta 0:00:00


In [ ]:
# Login to Hugging Face
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineGrained).
The tok

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
import json
from tqdm.auto import tqdm
import os
from datetime import datetime
import sys
import platform

# ============================================================================
# CONFIGURATION - Modify these settings
# ============================================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Alternative models for testing
MODEL_NAMES = [
               #"meta-llama/Llama-3.2-1B-Instruct",
               #"allenai/OLMo-2-0425-1B",
               #"Qwen/Qwen2.5-1.5B-Instruct",
               #"allenai/Olmo-3-7B-Instruct",
               #"meta-llama/Llama-3.1-8B-Instruct",
               #"Qwen/Qwen2.5-7B-Instruct"
               ]

MODEL_NAMES = [
               "Qwen/Qwen2.5-7B-Instruct",
               "meta-llama/Llama-3.1-8B-Instruct"
               ]

# GPU settings
# If True, will attempt to use the best available GPU (CUDA for NVIDIA, MPS for Apple Silicon)
# If False, will always use CPU regardless of available hardware
USE_GPU = True  # Set to False to force CPU-only execution
# USE_GPU = False  # Forcing CPU-only execution to compare performance and execution time

MAX_NEW_TOKENS = 1

# Quantization settings
# Options: 4, 8, or None (default is None for full precision)
#
# To enable quantization, change QUANTIZATION_BITS to one of the following:
#   QUANTIZATION_BITS = 4   # 4-bit quantization: ~1.5 GB memory (most memory efficient)
#   QUANTIZATION_BITS = 8   # 8-bit quantization: ~2.5 GB memory (balanced quality/memory)
#   QUANTIZATION_BITS = None  # No quantization: ~5 GB memory (full precision, best quality)
#
# Notes:
# - Quantization requires the 'bitsandbytes' package: pip install bitsandbytes
# - Quantization only works with CUDA (NVIDIA GPUs), not with Apple Metal (MPS)
# - If using Apple Silicon, quantization will be automatically disabled

QUANTIZATION_BITS = 8  # Change to 4 or 8 to enable quantization

# For quick testing, you can reduce this list
MMLU_SUBJECTS = [
    # "abstract_algebra", "anatomy",
    "astronomy", "business_ethics",
    # "clinical_knowledge", "college_biology", "college_chemistry",
    # "college_computer_science", "college_mathematics", "college_medicine",
    # "college_physics", "computer_security", "conceptual_physics",
    # "econometrics", "electrical_engineering", "elementary_mathematics",
    # "formal_logic", "global_facts", "high_school_biology",
    # "high_school_chemistry", "high_school_computer_science",
    # "high_school_european_history",
    # "high_school_geography",
    # "high_school_government_and_politics",
    # "high_school_macroeconomics",
    # "high_school_mathematics",
    # "high_school_microeconomics",
    # "high_school_physics",
    # "high_school_psychology",
    # "high_school_statistics",
    # "high_school_us_history",
    # "high_school_world_history",
    # "human_aging",
    # "human_sexuality",
    # "international_law",
    # "jurisprudence",
    # "logical_fallacies",
    # "machine_learning",
    # "management",
    # "marketing",
    # "medical_genetics",
    # "miscellaneous",
    # "moral_disputes",
    # "moral_scenarios",
    "nutrition", "philosophy", "prehistory", "professional_accounting",
    # "professional_law", "professional_medicine",
    # "professional_psychology",
    "public_relations", "security_studies", "sociology",
    "us_foreign_policy",
    # "virology", "world_religions"
]

def detect_device():
    """Detect the best available device (CUDA, MPS, or CPU)"""

    # If GPU is disabled, always use CPU
    if not USE_GPU:
        return "cpu"

    # Check for CUDA
    if torch.cuda.is_available():
        return "cuda"

    # Check for Apple Silicon with Metal
    if torch.backends.mps.is_available():
        # Check if we're actually on Apple ARM
        is_apple_arm = platform.system() == "Darwin" and platform.processor() == "arm"

        if is_apple_arm:
            # Metal is available but incompatible with quantization
            if QUANTIZATION_BITS is not None:
                print("\n" + "="*70)
                print("ERROR: Metal and Quantization Conflict")
                print("="*70)
                print(
                    "Metal Performance Shaders (MPS) is incompatible with quantization.")
                print(
                    f"You have USE_GPU = True and QUANTIZATION_BITS = {QUANTIZATION_BITS}")
                print("")
                print("Please choose one of the following options:")
                print("  1. Set USE_GPU = False to use CPU with quantization")
                print(
                    "  2. Set QUANTIZATION_BITS = None to use Metal without quantization")
                print("="*70 + "\n")
                sys.exit(1)
            return "mps"

    # Default to CPU
    return "cpu"


def check_environment():
    global QUANTIZATION_BITS
    """Check environment and dependencies"""
    print("="*70)
    print("Environment Check")
    print("="*70)

    # Check if in Colab
    try:
        import google.colab
        print("âœ“ Running in Google Colab")
        in_colab = True
    except:
        print("âœ“ Running locally (not in Colab)")
        in_colab = False

    # Check system info
    print(f"âœ“ Platform: {platform.system()} ({platform.machine()})")
    if platform.system() == "Darwin":
        print(f"âœ“ Processor: {platform.processor()}")

    # Detect and set device
    device = detect_device()

    # Check device
    if device == "cuda":
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"âœ“ GPU Available: {gpu_name}")
        print(f"âœ“ GPU Memory: {gpu_memory:.2f} GB")
    elif device == "mps":
        print("âœ“ Apple Metal (MPS) Available")
        print("âœ“ Using Metal Performance Shaders for GPU acceleration")
    else:
        print("âš ï¸  No GPU detected - running on CPU")

    # Check quantization support

    if QUANTIZATION_BITS is not None:
        try:
            import bitsandbytes
            print(
                f"âœ“ bitsandbytes installed - {QUANTIZATION_BITS}-bit quantization available")
        except ImportError:
            print(f"âŒ bitsandbytes NOT installed - cannot use quantization")
            sys.exit(1)
        if device == 'mps':
            print(f"âŒ Apple METAL is incompatible with quantization")
            print("âœ“ Quantization disabled - loading full precision model")
            QUANTIZATION_BITS = None
            sys.exit(1)
    else:
        print("âœ“ Quantization disabled - loading full precision model")

    # Check HF authentication
    try:
        from huggingface_hub import HfFolder
        token = HfFolder.get_token()
        if token:
            print("âœ“ Hugging Face authenticated")
        else:
            print("âš ï¸  No Hugging Face token found")
            print("Run: huggingface-cli login")
    except:
        print("âš ï¸  Could not check Hugging Face authentication")

    # Print configuration
    print("\n" + "="*70)
    print("Configuration")
    print("="*70)
    print(f"Model: {MODEL_NAME}")
    print(f"Device: {device}")
    if QUANTIZATION_BITS is not None:
        print(f"Quantization: {QUANTIZATION_BITS}-bit")
        if QUANTIZATION_BITS == 4:
            print(f"Expected memory: ~1.5 GB")
        elif QUANTIZATION_BITS == 8:
            print(f"Expected memory: ~2.5 GB")
    else:
        print(f"Quantization: None (full precision)")
        if device == "cuda":
            print(f"Expected memory: ~2.5 GB (FP16)")
        elif device == "mps":
            print(f"Expected memory: ~2.5 GB (FP16)")
        else:
            print(f"Expected memory: ~5 GB (FP32)")
    print(f"Number of subjects: {len(MMLU_SUBJECTS)}")

    print("="*70 + "\n")
    return in_colab, device


def get_quantization_config():
    """Create quantization config based on settings"""
    if QUANTIZATION_BITS is None:
        return None

    if QUANTIZATION_BITS == 4:
        # 4-bit quantization (most memory efficient)
        config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,  # Double quantization for extra compression
            bnb_4bit_quant_type="nf4"  # NormalFloat4 - better for LLMs
        )
        print("Using 4-bit quantization (NF4 + double quant)")
        print("Memory usage: ~1.5 GB")
    elif QUANTIZATION_BITS == 8:
        # 8-bit quantization (balanced)
        config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_threshold=6.0,
            llm_int8_has_fp16_weight=False
        )
        print("Using 8-bit quantization")
        print("Memory usage: ~2.5 GB")
    else:
        raise ValueError(
            f"Invalid QUANTIZATION_BITS: {QUANTIZATION_BITS}. Use 4, 8, or None")

    return config


def load_model_and_tokenizer(device):
    """Load Llama model with optional quantization"""
    print(f"\nLoading model {MODEL_NAME}...")
    print(f"Device: {device}")

    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        print("âœ“ Tokenizer loaded")

        # Get quantization config
        quant_config = get_quantization_config()

        # Load model
        print("Loading model (this may take 2-3 minutes)...")

        if quant_config is not None:
            # Quantized model loading (only works with CUDA)
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                quantization_config=quant_config,
                device_map="auto",
                low_cpu_mem_usage=True
            )
        else:
            # Non-quantized model loading
            if device == "cuda":
                model = AutoModelForCausalLM.from_pretrained(
                    MODEL_NAME,
                    dtype=torch.float16,
                    device_map="auto",
                    low_cpu_mem_usage=True
                )
            elif device == "mps":
                model = AutoModelForCausalLM.from_pretrained(
                    MODEL_NAME,
                    dtype=torch.float16,
                    low_cpu_mem_usage=True
                )
                model = model.to(device)
            else:  # CPU
                model = AutoModelForCausalLM.from_pretrained(
                    MODEL_NAME,
                    dtype=torch.float32,
                    low_cpu_mem_usage=True
                )
                model = model.to(device)

        model.eval()

        # Print model info
        print("âœ“ Model loaded successfully!")
        print(f"  Model device: {next(model.parameters()).device}")
        print(f"  Model dtype: {next(model.parameters()).dtype}")

        # Check memory usage
        if torch.cuda.is_available():
            memory_allocated = torch.cuda.memory_allocated(0) / 1e9
            memory_reserved = torch.cuda.memory_reserved(0) / 1e9
            print(
                f"  GPU Memory: {memory_allocated:.2f} GB allocated, {memory_reserved:.2f} GB reserved")

            # Check if using quantization
            if quant_config is not None:
                print(f"  Quantization: {QUANTIZATION_BITS}-bit active")
        elif device == "mps":
            print(f"  Running on Apple Metal (MPS)")

        return model, tokenizer

    except Exception as e:
        print(f"\nâŒ Error loading model: {e}")
        print("\nPossible causes:")
        print("1. No Hugging Face token - Run: huggingface-cli login")
        print("2. Llama license not accepted - Visit: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct")
        print("3. bitsandbytes not installed - Run: pip install bitsandbytes")
        print("4. Out of memory - Try 4-bit quantization or smaller model")
        raise


def format_mmlu_prompt(question, choices):
    """Format MMLU question as multiple choice"""
    choice_labels = ["A", "B", "C", "D"]
    prompt = f"{question}\n\n"
    for label, choice in zip(choice_labels, choices):
        prompt += f"{label}. {choice}\n"
    prompt += "\nAnswer:"
    return prompt


def get_model_prediction(model, tokenizer, prompt):
    """Get model's prediction for multiple-choice question"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
            temperature=1.0
        )

    generated_text = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    answer = generated_text.strip()[:1].upper()

    if answer not in ["A", "B", "C", "D"]:
        for char in generated_text.upper():
            if char in ["A", "B", "C", "D"]:
                answer = char
                break
        else:
            answer = "A"

    return answer


def evaluate_subject(model, tokenizer, subject):
    """Evaluate model on a specific MMLU subject"""
    print(f"\n{'='*70}")
    print(f"Evaluating subject: {subject}")
    print(f"{'='*70}")

    try:
        dataset = load_dataset("cais/mmlu", subject, split="test")
    except Exception as e:
        print(f"âŒ Error loading subject {subject}: {e}")
        return None

    correct = 0
    total = 0

    for example in tqdm(dataset, desc=f"Testing {subject}", leave=True):
        question = example["question"]
        choices = example["choices"]
        correct_answer_idx = example["answer"]
        correct_answer = ["A", "B", "C", "D"][correct_answer_idx]

        prompt = format_mmlu_prompt(question, choices)
        predicted_answer = get_model_prediction(model, tokenizer, prompt)

        choice_dict = {"A": choices[0], "B": choices[1],
                       "C": choices[2], "D": choices[3]}

        print(f"\nQuestion: {question}")
        print(f"Predicted Answer: {choice_dict.get(predicted_answer, 'N/A')}")
        print(
            f"Status: {'Correct' if predicted_answer == correct_answer else 'Incorrect'}")

        if predicted_answer == correct_answer:
            correct += 1
        total += 1

    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"âœ“ Result: {correct}/{total} correct = {accuracy:.2f}%")

    return {
        "subject": subject,
        "correct": correct,
        "total": total,
        "accuracy": accuracy
    }


def main():
    """Main evaluation function"""
    print("\n" + "="*70)
    print(f"{MODEL_NAME} MMLU Evaluation (Quantized)")
    print("="*70 + "\n")

    # Check environment
    in_colab, device = check_environment()

    # Load model
    model, tokenizer = load_model_and_tokenizer(device)

    # Evaluate
    results = []
    total_correct = 0
    total_questions = 0

    print(f"\n{'='*70}")
    print(f"Starting evaluation on {len(MMLU_SUBJECTS)} subjects")
    print(f"{'='*70}\n")

    start_time = datetime.now()

    for i, subject in enumerate(MMLU_SUBJECTS, 1):
        print(f"\nProgress: {i}/{len(MMLU_SUBJECTS)} subjects")
        result = evaluate_subject(model, tokenizer, subject)
        if result:
            results.append(result)
            total_correct += result["correct"]
            total_questions += result["total"]

    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()

    # Calculate overall accuracy
    overall_accuracy = (total_correct / total_questions *
                        100) if total_questions > 0 else 0

    # Print summary
    print("\n" + "="*70)
    print("EVALUATION SUMMARY")
    print("="*70)
    print(f"Model: {MODEL_NAME}")
    print(
        f"Quantization: {QUANTIZATION_BITS}-bit" if QUANTIZATION_BITS else "None (full precision)")
    print(f"Total Subjects: {len(results)}")
    print(f"Total Questions: {total_questions}")
    print(f"Total Correct: {total_correct}")
    print(f"Overall Accuracy: {overall_accuracy:.2f}%")
    print(f"Duration: {duration/60:.1f} minutes")
    print("="*70)

    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    quant_suffix = f"_{QUANTIZATION_BITS}bit" if QUANTIZATION_BITS else "_full"
    output_file = f"{MODEL_NAME.replace('/', '_')}_mmlu_results{quant_suffix}_{timestamp}.json"

    output_data = {
        "model": MODEL_NAME,
        "quantization_bits": QUANTIZATION_BITS,
        "timestamp": timestamp,
        "device": str(device),
        "duration_seconds": duration,
        "overall_accuracy": overall_accuracy,
        "total_correct": total_correct,
        "total_questions": total_questions,
        "subject_results": results
    }

    with open(output_file, "w") as f:
        json.dump(output_data, f, indent=2)

    print(f"\nâœ“ Results saved to: {output_file}")

    # Print top/bottom subjects
    if len(results) > 0:
        sorted_results = sorted(
            results, key=lambda x: x["accuracy"], reverse=True)

        print("\nðŸ“Š Top 5 Subjects:")
        for i, result in enumerate(sorted_results[:5], 1):
            print(f"  {i}. {result['subject']}: {result['accuracy']:.2f}%")

        print("\nðŸ“‰ Bottom 5 Subjects:")
        for i, result in enumerate(sorted_results[-5:], 1):
            print(f"  {i}. {result['subject']}: {result['accuracy']:.2f}%")

    # Colab-specific instructions
    if in_colab:
        print("\n" + "="*70)
        print("ðŸ’¾ To download results in Colab:")
        print("="*70)
        print(f"from google.colab import files")
        print(f"files.download('{output_file}')")

    print("\nâœ… Evaluation complete!")
    return output_file


def testing_multiple_models():
    """Test multiple models on MMLU"""
    results = {}
    for model_name in MODEL_NAMES:
        global MODEL_NAME
        MODEL_NAME = model_name
        print(f"\n\n{'#'*80}")
        print(f"Testing model: {MODEL_NAME}")
        print(f"{'#'*80}\n")
        output_file = main()
        results[model_name] = output_file

    # save results in a json file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    summary_file = f"mmlu_multiple_models_results_{timestamp}.json"
    with open(summary_file, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nâœ“ Summary of multiple model results saved to: {summary_file}")


if __name__ == "__main__":
    try:
        # output_file = main()
        testing_multiple_models()
    except KeyboardInterrupt:
        print("\n\nâš ï¸  Evaluation interrupted by user")
    except Exception as e:
        print(f"\nâŒ Error during evaluation: {e}")
        import traceback
        traceback.print_exc()




################################################################################
Testing model: Qwen/Qwen2.5-7B-Instruct
################################################################################


Qwen/Qwen2.5-7B-Instruct MMLU Evaluation (Quantized)

Environment Check
âœ“ Running in Google Colab
âœ“ Platform: Linux (x86_64)
âœ“ GPU Available: Tesla T4
âœ“ GPU Memory: 15.83 GB
âœ“ bitsandbytes installed - 8-bit quantization available
âœ“ Hugging Face authenticated

Configuration
Model: Qwen/Qwen2.5-7B-Instruct
Device: cuda
Quantization: 8-bit
Expected memory: ~2.5 GB
Number of subjects: 10


Loading model Qwen/Qwen2.5-7B-Instruct...
Device: cuda
âœ“ Tokenizer loaded
Using 8-bit quantization
Memory usage: ~2.5 GB
Loading model (this may take 2-3 minutes)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

âœ“ Model loaded successfully!
  Model device: cuda:0
  Model dtype: torch.float16
  GPU Memory: 8.73 GB allocated, 14.27 GB reserved
  Quantization: 8-bit active

Starting evaluation on 10 subjects


Progress: 1/10 subjects

Evaluating subject: astronomy


Testing astronomy:   0%|          | 0/152 [00:00<?, ?it/s]


Question: What is true for a type-Ia ("type one-a") supernova?
Predicted Answer: This type occurs in binary systems.
Status: Correct

Question: If you know both the actual brightness of an object and its apparent brightness from your location then with no other information you can estimate:
Predicted Answer: Its distance from you
Status: Correct

Question: Why is the sky blue?
Predicted Answer: Because the atmosphere preferentially scatters short wavelengths.
Status: Correct

Question: You’ve made a scientific theory that there is an attractive force between all objects. When will your theory be proven to be correct?
Predicted Answer: The first time you drop a bowling ball and it falls to the ground proving your hypothesis.
Status: Incorrect

Question: Which of the following is/are true?
Predicted Answer: A and D
Status: Correct

Question: A comet of mass m impacts the earth (mass M radius R) at the minimum impact speed. What is the expression for the total energy released in the impa

...


Question: Which of these principles is not an element of the responsibility to protect?
Predicted Answer: The responsibility to remain sovereign.
Status: Correct

Question: Can environmental changes be reconciled with national security interests?
Predicted Answer: All of these options.
Status: Incorrect

Question: Which of the following statements could be described as a liberal perspective on future energy security?
Predicted Answer: The global economy is interconnected, ensuring that energy security for one is dependent upon energy security for all. Thus all core powers have the same interests in maintaining and extending the conditions under which this market operates. As long as this economic order exists, conflict between major powers over energy reserves is highly unlikely.
Status: Correct

Question: Which of the following is a common criticism of the human security concept?
Predicted Answer: Human security is too broad.
Status: Incorrect

Question: In what ways have governments

Testing sociology:   0%|          | 0/201 [00:00<?, ?it/s]


Question: Mass-society theory suggests that:
Predicted Answer: the media manipulate 'the masses' as vulnerable, passive consumers
Status: Correct

Question: The ecological approach to urban sociology involved studying:
Predicted Answer: the forms of wildlife and natural habitats that could be found on the edges of the city
Status: Incorrect

Question: Becker proclaimed that cannabis use was:
Predicted Answer: increasing throughout all sections of youth in the 1970s
Status: Incorrect

Question: The middle classes that developed over the nineteenth century were:
Predicted Answer: all of the above
Status: Correct

Question: Technological forms of surveillance have made it easier to:
Predicted Answer: all of the above
Status: Correct

Question: The concept of gemeinschaft developed by Ferdinand Tönnies describes basically the same relational characteristics as
Predicted Answer: mechanical solidarity
Status: Correct

Question: There was a growth in income inequality in the 1980s because:
P

Testing us_foreign_policy:   0%|          | 0/100 [00:00<?, ?it/s]


Question: What is the structure of the United Nations Security Council?
Predicted Answer: 5 permanent members with veto power, 10 rotating members with no veto power
Status: Correct

Question: What was the significance of the Gulf of Tonkin resolution?
Predicted Answer: It allowed the US to intensify its involvement in Vietnam
Status: Correct

Question: Which is not a nonstate actor that poses a threat to the United States?
Predicted Answer: China
Status: Correct

Question: Who was the first American president to visit communist China?
Predicted Answer: Richard Nixon
Status: Correct

Question: The Strategic Arms Reduction Treaty was the first accord
Predicted Answer: mandating the elimination of many long-range nuclear missiles.
Status: Correct

Question: What were the implications of the Cold War for American exceptionalism?
Predicted Answer: Both b and c
Status: Correct

Question: Why did Franklin D. Roosevelt initially favour an 'isolationist' stance on the part of the US during th